# Leadgen — load, chọn biến, train

Notebook chạy cùng pipeline với `train.py`, nhưng **mọi tham số nằm trong notebook** (mục 1) thay vì `configs/leadgen.yaml`.
Sửa ô config rồi chạy lại từ mục 2.

- Mở notebook tại thư mục `train_model` (kernel: `/home/hitech/phuoc_btc/.venv/bin/python`).
- Mặc định ghi ra `outputs/leadgen_nb/` và `models/leadgen_nb/` để **không ghi đè** model chính thức (`models/leadgen/`).
- Config thực tế của mỗi lần chạy được lưu ở `<output_dir>/config_used.yaml`.

In [20]:
import json
import sys
import time
from pathlib import Path

REPO = Path.cwd()
assert (REPO / "src" / "features.py").exists(), "Mở notebook tại thư mục train_model"
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from catboost import CatBoostClassifier, Pool
from IPython.display import Image, Markdown, display

from src import evaluate as ev
from src.data import load_features, load_labels, prepare_matrix
from src.features import METHODS, cat_features_of, select_features
from src.tuning import tune
from train import optuna_plot

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

## 1. Config

### 1.1 Nhãn, chia tập, MinIO

In [21]:
RUN_NAME = "leadgen"

LABEL = {
    "file": "/home/hitech/repo-vtp/services/jupyter/notebooks/vtp_v2/src/notebooks/merged_customer_application_fpd.xlsx",
    "sheet": "Merged",
    "key": "cus_id",
    "snapshot_col": "date",      # ngày snapshot WL (YYYY-MM-01)
    "positive_col": "month",     # có giá trị = shop đã nộp hồ sơ trong tháng snapshot
    "snapshots": [20260501, 20260601, 20260701, 20260801],
}

SPLIT = {                        # out-of-time
    "train": [20260501, 20260601],
    "test": [20260701, 20260801],
}

MINIO = {
    "connect_yaml": "/home/hitech/repo-vtp/services/jupyter/notebooks/vtp_v2/src/config/connect.yaml",
    "endpoint": "127.0.0.1:9990",  # endpoint từ host; connect.yaml ghi hostname docker
    "bucket": None,                # None = lấy từ connect_yaml
}

### 1.2 Feature: nguồn, cột loại, cột categorical

Mỗi thư mục S3 là một **group** feature. Đổi danh sách nguồn hoặc cột loại thì cache trong `data/` tự tạo mới.

In [ ]:
FEATURE_SOURCES = [
    "features/demographic/final",
    "features/gmv/final",
    "features/location/l1m",
    "features/sales/l1m",
    "features/sales/l6m",
    "features/trend/final",
    "features/vct/l1m",
    "features/vct/l3m",
    "features/cus_base/l1m",
    "features/cus_base/l6m",
    "features/cuoc_trongluong/l1m",
    "features/cuoc_trongluong/l6m",
]

EXCLUDE_COLUMNS = [
    # metadata, ngày thô, hằng số theo tháng
    "quarter",
    "duration_group",
    "dob",
    "ngay_hoptac",
    "population_cod_ship_rev_gmv_ratio_x",
    "population_cod_ship_rev_gmv_ratio_y",
    # Điều kiện vào WL: gần như luôn = 1 trong WL; riêng snapshot 20260601 trống 100%
    # (trend/final tháng đó không có cột) -> model học ra tháng thay vì hành vi shop.
    "delivered_orders_every_month_4m",
    "delivered_orders_every_month_3m",
    "delivered_orders_every_month_2m",
]

CATEGORICAL_COLUMNS = ["top1_location_l1m"]

### 1.3 Chọn biến

`STEPS` chạy lần lượt, mỗi bước nhận tập biến còn lại của bước trước. Mỗi bước là `{"method": ..., <tham số>}`.

| method | Ý nghĩa | Luật chọn mặc định |
|---|---|---|
| `stat_filter` | missing, constant, low_variance, duplicate, Spearman (ngưỡng ở `STAT_FILTER`) | — |
| `group_importance` | CatBoost CV từng group, cộng dồn importance | `cum_importance` |
| `cum_importance` | CatBoost CV toàn bộ biến, cộng dồn importance | `cum_importance` |
| `permutation` | AUC giảm khi xáo trộn biến (RandomForest) | score > 0 |
| `single_auc` | AUC CV của DecisionTree chỉ một biến | ≥ 0,52 |
| `beat_random` | importance RandomForest > cột ngẫu nhiên | — |
| `l1`, `l2` | \|hệ số\| LogisticRegression (categorical cho qua) | ≥ 0,1 × trung bình |
| `mutual_info` | mutual information với nhãn | top 75% |
| `target_corr` | \|Pearson\| với nhãn (categorical cho qua) | ≥ 0,1 và top 10 |
| `rfe` | RFE DecisionTree (`k`: số biến hoặc tỷ lệ, `step`) | k = 0,5 |
| `vote` | chạy nhiều phương pháp, giữ biến được ≥ `min_share` số phương pháp chọn | min_share = 0,5 |

Tham số chung mọi bước: `sample_rows` (chạy trên mẫu theo shop), `min_score`, `min_score_rel_mean`, `top_k`, `top_pct` (`None` = tắt luật mặc định).

In [ ]:
STAT_FILTER = {
    "max_missing": 0.95,        # bỏ nếu thiếu > 95%                                   (tắt: 1.0)
    "max_dominant": 0.995,      # bỏ nếu một giá trị chiếm > 99,5%                      (tắt: 1.0)
    "min_scaled_var": 0.001,    # bỏ nếu phương sai (cắt p1–p99, MinMaxScaler) < 0,001   (tắt: 0)
    "max_equal_share": 0.99,    # hai cột giống hệt ở >= 99% số dòng: bỏ cột AUC thấp hơn (tắt: 1.01)
    "max_spearman": 0.95,       # cặp |Spearman| > 0,95: bỏ cột AUC đơn biến thấp hơn     (tắt: 1.0)
    "corr_sample": 50000,       # số dòng mẫu cho duplicate và Spearman
}

STEPS = [
    {"method": "stat_filter"},
    {"method": "group_importance", "cum_importance": 0.95},
    {"method": "cum_importance", "cum_importance": 0.85, "min_features": 10,
     "cv_folds": 5, "catboost": {"iterations": 1000}},
]

# # Ví dụ: bỏ phiếu giữa 3 phương pháp trước bước importance cuối
# STEPS = [
#     {"method": "stat_filter"},
#     {"method": "vote", "min_share": 0.67, "methods": [
#         {"method": "permutation", "n_estimators": 300},
#         {"method": "beat_random", "n_estimators": 300},
#         {"method": "mutual_info", "sample_rows": 50000},
#     ]},
#     {"method": "cum_importance"},
# ]

SELECTION = {
    "cum_importance": 0.90,     # giữ biến tới khi cộng dồn importance >= 90%
    "min_features_group": 1,
    "min_features_final": 5,
    "cv_folds": 3,
    "catboost": {               # tham số nhanh cho các bước dùng CatBoost
        "iterations": 600,
        "learning_rate": 0.05,
        "depth": 6,
        "early_stopping_rounds": 50,
    },
    "steps": STEPS,
}
print("Phương pháp có sẵn:", sorted(METHODS))

Phương pháp có sẵn: ['beat_random', 'cum_importance', 'group_importance', 'l1', 'l2', 'mutual_info', 'permutation', 'rfe', 'single_auc', 'stat_filter', 'target_corr', 'vote']


### 1.4 Optuna, đánh giá, đường dẫn

In [24]:
OPTUNA = {
    "n_trials": 5,
    "cv_folds": 3,
    "timeout_sec": 3600,
    "max_iterations": 2000,
    "early_stopping_rounds": 100,
}

SEED = 42
THREAD_COUNT = 14

EVAL = {
    "n_bins": 10,               # precision theo decile
    "shap_top": 10,
    "shap_sample": 5000,
    "psi_bins": 10,
}

PATHS = {
    "cache_dir": "data",                  # cache dữ liệu đã merge (không commit)
    "output_dir": "outputs/leadgen_nb",   # đổi thành outputs/leadgen để ghi đè kết quả chính thức
    "model_dir": "models/leadgen_nb",
}

In [25]:
cfg = {
    "run_name": RUN_NAME, "label": LABEL, "split": SPLIT, "minio": MINIO,
    "feature_sources": FEATURE_SOURCES, "exclude_columns": EXCLUDE_COLUMNS, "categorical_columns": CATEGORICAL_COLUMNS,
    "stat_filter": STAT_FILTER, "selection": SELECTION, "optuna": OPTUNA,
    "seed": SEED, "thread_count": THREAD_COUNT, "eval": EVAL, "paths": PATHS,
}
out = Path(PATHS["output_dir"]); out.mkdir(parents=True, exist_ok=True)
mdir = Path(PATHS["model_dir"]); mdir.mkdir(parents=True, exist_ok=True)
(out / "config_used.yaml").write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False), encoding="utf-8")
print(f"config -> {out / 'config_used.yaml'}")

config -> outputs/leadgen_nb/config_used.yaml


## 2. Load dữ liệu

Nhãn từ Excel, feature từ MinIO (lần đầu đọc S3, các lần sau đọc cache `data/`).

In [26]:
t0 = time.time()
labels = load_labels(cfg)
df, feature_group = load_features(cfg, labels)
df = df.merge(labels, on=["cus_id", "date"], how="left", validate="one_to_one")
df["split"] = np.where(df["date"].isin(SPLIT["train"]), "train",
                       np.where(df["date"].isin(SPLIT["test"]), "test", "unused"))
df = df[df["split"] != "unused"].reset_index(drop=True)

features_all = list(feature_group)
X = prepare_matrix(df, features_all, CATEGORICAL_COLUMNS)
cats_all = cat_features_of(X, features_all)
tr = (df["split"] == "train").to_numpy()
te = ~tr
y = df["label"]
print(f"rows train={tr.sum():,} test={te.sum():,} | features={len(features_all)} | categorical={cats_all}")

rows train=106,077 test=101,049 | features=210 | categorical=['top1_location_l1m']


In [27]:
ev.label_rate(df, out)
display(pd.read_csv(out / "label_rate_by_month.csv"))
display(pd.Series(feature_group).value_counts().rename("số biến").to_frame())

,date,split,n,n_pos,label_rate
0,20260501,train,53427,205,0.003837
1,20260601,train,52650,467,0.008870
2,20260701,test,51039,340,0.006662
3,20260801,test,50010,243,0.004859


,số biến
sales/l6m,46
vct/l3m,46
cuoc_trongluong/l6m,27
cus_base/l6m,25
vct/l1m,23
cuoc_trongluong/l1m,16
sales/l1m,9
cus_base/l1m,8
gmv/final,4
demographic/final,2


## 3. Chọn biến (chỉ trên tập train)

In [28]:
Xtr = X[tr].reset_index(drop=True)
ytr = y[tr].reset_index(drop=True)
gtr = df.loc[tr, "cus_id"].reset_index(drop=True)

sel = select_features(Xtr, ytr, gtr, features_all, feature_group, cfg)
features = sel["final"]
cats = [c for c in features if c in cats_all]
for name, log in sel["logs"]:
    log.to_csv(out / f"selection_{name}.csv", index=False)
display(pd.DataFrame(sel["counts"], columns=["bước", "số biến còn lại"]))

[select] 1_stat_filter: 210 -> 95
[select]   vote 1_permutation: chọn 79/95
[select]   vote 2_beat_random: chọn 92/95
[select]   vote 3_mutual_info: chọn 72/95
[select] 2_vote: 95 -> 61
[select] 3_cum_importance: 61 -> 44


,bước,số biến còn lại
0,input,210
1,1_stat_filter,95
2,2_vote,61
3,3_cum_importance,44


In [29]:
# Xem log từng bước: logs["1_stat_filter"], logs["2_group_importance"], ...
logs = dict(sel["logs"])
for name, log in logs.items():
    if name.endswith("stat_filter"):
        reason = log["drop_reason"].str.split(r"[ >]").str[0].replace("", "kept")
        display(Markdown(f"**{name}** — lý do loại"), reason.value_counts().to_frame("số biến"))
last = sel["logs"][-1][0]
display(Markdown(f"**{last}** — biến được chọn"), logs[last][logs[last]["selected"]].head(60))

**1_stat_filter** — lý do loại

,số biến
drop_reason,
kept,95
spearman,91
duplicate,24


**3_cum_importance** — biến được chọn

,feature,score,cum_share,selected,cv_auc,group
0,solan_rut_cod_sum_l3m,18.513326,0.185133,True,0.882904,vct/l3m
1,solan_rut_cod_l1m,5.042674,0.235560,True,0.882904,vct/l1m
2,is_cod_x,4.567378,0.281234,True,0.882904,gmv/final
3,trongluong_max_min_ratio_l1m,2.973758,0.310971,True,0.882904,cuoc_trongluong/l1m
4,thu_ho_min_l6m,2.865144,0.339623,True,0.882904,sales/l6m
5,ty_le_cuoc_tren_thuho_mean_l3m,2.521630,0.364839,True,0.882904,vct/l3m
6,ty_le_don_co_cod_l3m,2.360641,0.388446,True,0.882904,vct/l3m
7,ty_le_don_co_cod_l1m,2.304618,0.411492,True,0.882904,vct/l1m
8,trongluong_max_min_ratio_min_l6m,2.167744,0.433169,True,0.882904,cuoc_trongluong/l6m
9,trongluong_max_to_avg_ratio_l1m,1.974478,0.452914,True,0.882904,cuoc_trongluong/l1m


## 4. Optuna

In [30]:
tun = tune(Xtr, ytr, gtr, features, cfg)
params = tun["params"]
tun["trials"].to_csv(out / "optuna_trials.csv", index=False)
optuna_plot(tun["trials"], out)
(out / "best_params.json").write_text(json.dumps({**params, "cv_auc": tun["cv_auc"]}, indent=2))
print(f"AUC CV tốt nhất {tun['cv_auc']:.4f}")
params

[optuna] trial   0 auc=0.8829 best=0.8829
[optuna] trial   1 auc=0.8792 best=0.8829
[optuna] trial   2 auc=0.8829 best=0.8829
[optuna] trial   3 auc=0.8836 best=0.8836
[optuna] trial   4 auc=0.8649 best=0.8836
AUC CV tốt nhất 0.8836


{'learning_rate': 0.03920673972242137,
 'depth': 7,
 'l2_leaf_reg': 1.9721610970573997,
 'random_strength': 2.571172192068058,
 'bagging_temperature': 2.9620728443102124,
 'border_count': 128,
 'iterations': 264}

## 5. Model cuối (toàn bộ train)

In [31]:
model = CatBoostClassifier(**params, eval_metric="AUC", random_seed=SEED, verbose=False,
                           thread_count=THREAD_COUNT, allow_writing_files=False)
model.fit(Pool(Xtr[features], ytr, cat_features=cats))
model.save_model(str(mdir / "model.cbm"))
print(f"model -> {mdir / 'model.cbm'}")

model -> models/leadgen_nb/model.cbm


## 6. Đánh giá

Train dùng dự báo OOF của CV, test dùng model cuối (out-of-time).

In [32]:
df["score"] = np.nan
df.loc[tr, "score"] = tun["oof"]
df.loc[te, "score"] = model.predict_proba(X.loc[te, features])[:, 1]
df["score_insample"] = model.predict_proba(X[features])[:, 1]

aucs = ev.auc_by_month(df[["date", "split", "label", "score"]], out)
test_df = df[te][["date", "label", "score"]].reset_index(drop=True)
prec = ev.precision_tables(test_df, EVAL["n_bins"], out)
display(aucs)
print("PR-AUC model cuối:", ", ".join(f"{r.date} {r.pr_auc:.4f} (×{r.pr_lift:.2f} tỷ lệ nhãn)"
                                     for r in aucs[aucs["date"].astype(str).str.startswith("all_")].itertuples()))
display(Image(filename=out / "auc_gini_by_month.png"), Image(filename=out / "pr_train_test.png"))
display(prec[prec["date"] == "all_test"])

,date,split,n,n_pos,auc,gini
0,20260501,train,53427,205,0.908331,0.816662
1,20260601,train,52650,467,0.866229,0.732458
2,20260701,test,51039,340,0.883976,0.767953
3,20260801,test,50010,243,0.866220,0.732440
4,all_test,test,101049,583,0.876656,0.753311
5,all_train,train,106077,672,0.880114,0.760228


,date,bin,n,n_pos,score_min,score_max,precision,cum_n,cum_pos,cum_precision,cum_recall,lift,cum_lift
0,all_test,1,10105,372,0.017039,0.758677,0.036813,10105,372,0.036813,0.638079,6.380726,6.380726
1,all_test,2,10105,111,0.004226,0.017039,0.010985,20210,483,0.023899,0.828473,1.903926,4.142326
2,all_test,3,10105,34,0.002749,0.004226,0.003365,30315,517,0.017054,0.886792,0.583185,2.955946
3,all_test,4,10105,14,0.002124,0.002749,0.001385,40420,531,0.013137,0.910806,0.240135,2.276993
4,all_test,5,10105,18,0.001684,0.002124,0.001781,50525,549,0.010866,0.941681,0.308745,1.883343
5,all_test,6,10104,11,0.001323,0.001684,0.001089,60629,560,0.009237,0.960549,0.188696,1.600925
6,all_test,7,10105,12,0.000976,0.001323,0.001188,70734,572,0.008087,0.981132,0.205830,1.401623
7,all_test,8,10105,7,0.000608,0.000976,0.000693,80839,579,0.007162,0.993139,0.120067,1.241427
8,all_test,9,10105,3,0.000324,0.000608,0.000297,90944,582,0.006400,0.998285,0.051457,1.109206
9,all_test,10,10105,1,0.000057,0.000324,0.000099,101049,583,0.005769,1.000000,0.017152,1.000000


In [33]:
psi_train = X[tr][features].assign(score=df.loc[tr, "score_insample"].to_numpy())
psi_test = X[te][features].assign(score=df.loc[te, "score_insample"].to_numpy(), date=df.loc[te, "date"].to_numpy())
psi_s, psi_f = ev.psi_report(psi_train, psi_test, features, cats, EVAL["psi_bins"], out)

fi, gi = ev.importance_report(model, features, feature_group, out)
shap_t = ev.shap_report(model, X[te].reset_index(drop=True), features, cats, fi,
                        EVAL["shap_top"], EVAL["shap_sample"], SEED, out)
display(psi_s, gi, fi.head(20))

,date,psi_score
0,20260701,0.002797
1,20260801,0.005287
2,all_test,0.003715


,group,n_features,importance,share
0,vct/l3m,10,34.636494,0.346365
1,cuoc_trongluong/l6m,10,19.567879,0.195679
2,vct/l1m,7,16.936772,0.169368
3,cus_base/l6m,4,6.556666,0.065567
4,cuoc_trongluong/l1m,3,6.357510,0.063575
5,sales/l6m,2,3.543445,0.035434
6,sales/l1m,2,3.473223,0.034732
7,demographic/final,2,3.333866,0.033339
8,location/l1m,1,2.255619,0.022556
9,gmv/final,2,1.916132,0.019161


,feature,importance,group,share
0,solan_rut_cod_sum_l3m,12.347143,vct/l3m,0.123471
1,solan_rut_cod_l1m,4.108748,vct/l1m,0.041087
2,ty_le_don_co_cod_l3m,4.030507,vct/l3m,0.040305
3,ty_le_don_co_cod_l1m,3.124147,vct/l1m,0.031241
4,avg_thuho_per_day_l3m,2.925946,vct/l3m,0.029259
5,trongluong_max_min_ratio_min_l6m,2.684238,cuoc_trongluong/l6m,0.026842
6,trongluong_max_to_avg_ratio_l1m,2.682740,cuoc_trongluong/l1m,0.026827
7,ty_le_cuoc_tren_thuho_mean_l3m,2.643862,vct/l3m,0.026439
8,avg_tiencod_per_don_l1m,2.427875,vct/l1m,0.024279
9,ty_le_von_tren_thuho_mean_l3m,2.403731,vct/l3m,0.024037


## 7. Lưu meta, bảng báo cáo, tóm tắt

In [34]:
cut = np.quantile(df.loc[te, "score"], np.linspace(0, 1, EVAL["n_bins"] + 1))[1:-1].tolist()
meta = {
    "run_name": RUN_NAME, "trained_at": time.strftime("%Y-%m-%d %H:%M"),
    "features": features, "cat_features": cats,
    "feature_group": {f: feature_group[f] for f in features},
    "feature_sources": FEATURE_SOURCES, "exclude_columns": EXCLUDE_COLUMNS,
    "categorical_columns": CATEGORICAL_COLUMNS,
    "train_snapshots": SPLIT["train"], "test_snapshots": SPLIT["test"],
    "params": params, "cv_auc": tun["cv_auc"],
    "test_auc": float(aucs.loc[aucs["date"] == "all_test", "auc"].iloc[0]),
    "test_pr_auc": float(aucs.loc[aucs["date"] == "all_test", "pr_auc"].iloc[0]),
    "decile_cutpoints_test": cut,
}
(mdir / "model_meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2))

with pd.ExcelWriter(out / "report_tables.xlsx") as w:
    pd.read_csv(out / "label_rate_by_month.csv").to_excel(w, sheet_name="label_rate", index=False)
    aucs.to_excel(w, sheet_name="auc_gini_by_month", index=False)
    prec.to_excel(w, sheet_name="precision_decile_test", index=False)
    psi_s.to_excel(w, sheet_name="psi_score", index=False)
    psi_f.to_excel(w, sheet_name="psi_features", index=False)
    fi.to_excel(w, sheet_name="importance_feature", index=False)
    gi.to_excel(w, sheet_name="importance_group", index=False)
    shap_t.to_excel(w, sheet_name="shap_top10", index=False)
    for name, log in sel["logs"]:
        log.to_excel(w, sheet_name=f"sel_{name}"[:31], index=False)
    tun["trials"].to_excel(w, sheet_name="optuna_trials", index=False)

steps = " → ".join(f"{name.split('_', 1)[1]} {n}" for name, n in sel["counts"][1:])
top_bin = prec[(prec["date"] == "all_test") & (prec["bin"] == 1)].iloc[0]
lines = [
    f"# Leadgen model — {meta['trained_at']}", "",
    f"- Nhãn: shop trong WL nộp hồ sơ trong tháng snapshot. Train {SPLIT['train']}, test {SPLIT['test']} (out-of-time).",
    f"- Chọn biến: {len(features_all)} → {steps}: **{len(features)} biến**.",
    f"- Optuna {len(tun['trials'])} trial, AUC CV train tốt nhất {tun['cv_auc']:.4f}. Tham số: `{json.dumps(params)}`",
    "", "| Tháng | Tập | n | Nộp hồ sơ | AUC | Gini | PR-AUC | PR-AUC / tỷ lệ nhãn |", "|---|---|---|---|---|---|---|---|",
]
for r in aucs.itertuples():
    lines.append(f"| {r.date} | {r.split} | {r.n:,} | {r.n_pos:,} | {r.auc:.4f} | {r.gini:.4f} | {r.pr_auc:.4f} | ×{r.pr_lift:.2f} |")
lines += ["", f"- Bin 1 (10% điểm cao nhất) trên test: precision {top_bin.precision:.2%}, lift ×{top_bin.lift:.2f}, "
              f"bắt {top_bin.cum_recall:.1%} số shop nộp hồ sơ.",
          "- PSI score: " + ", ".join(f"{r.date} {r.psi_score:.4f}" for r in psi_s.itertuples()),
          f"- Thời gian chạy: {(time.time() - t0) / 60:.1f} phút."]
(out / "summary.md").write_text("\n".join(lines), encoding="utf-8")
display(Markdown("\n".join(lines)))

# Leadgen model — 2026-10-25 11:27

- Nhãn: shop trong WL nộp hồ sơ trong tháng snapshot. Train [20260501, 20260601], test [20260701, 20260801] (out-of-time).
- Chọn biến: 210 → stat_filter 95 → vote 61 → cum_importance 44: **44 biến**.
- Optuna 5 trial, AUC CV train tốt nhất 0.8836. Tham số: `{"learning_rate": 0.03920673972242137, "depth": 7, "l2_leaf_reg": 1.9721610970573997, "random_strength": 2.571172192068058, "bagging_temperature": 2.9620728443102124, "border_count": 128, "iterations": 264}`

| Tháng | Tập | n | Nộp hồ sơ | AUC | Gini |
|---|---|---|---|---|---|
| 20260501 | train | 53,427 | 205 | 0.9083 | 0.8167 |
| 20260601 | train | 52,650 | 467 | 0.8662 | 0.7325 |
| 20260701 | test | 51,039 | 340 | 0.8840 | 0.7680 |
| 20260801 | test | 50,010 | 243 | 0.8662 | 0.7324 |
| all_test | test | 101,049 | 583 | 0.8767 | 0.7533 |
| all_train | train | 106,077 | 672 | 0.8801 | 0.7602 |

- Bin 1 (10% điểm cao nhất) trên test: precision 3.68%, lift ×6.38, bắt 63.8% số shop nộp hồ sơ.
- PSI score: 20260701 0.0028, 20260801 0.0053, all_test 0.0037
- Thời gian chạy: 10.1 phút.